# 📓 Exemplo 10: Funções e um laço sobre vários lugares

**Missão 9** · [![Abrir no Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/fcoliveira-utfpr/change_climate/blob/main/code/notebooks_exemplo/10_funcoes_e_capitais.ipynb)

> **Pergunta:** como aplicar o mesmo método a muitos lugares sem repetir código?

### 🧭 Como usar este notebook
1. 💾 **Faça uma cópia:** menu **Arquivo → Salvar uma cópia no Drive**. Mova a cópia para a pasta
   `mudanca_climatica_br/notebooks/`. Assim você pode mudar tudo sem estragar o original.
2. ▶️ **Rode uma célula por vez** (botão ▶ ou **Shift + Enter**) e **leia os comentários** (as linhas com `#`).
3. 🧪 Os exemplos usam **Santa Helena-PR** e **Curitiba-PR**.
4. ✍️ No fim, a seção **Sua vez** pede para você **replicar o método para as capitais do Brasil**.

In [ ]:
# Instala as bibliotecas que não vêm no Colab (só precisa rodar uma vez por sessão)
!pip install agrometeorologiapy pymannkendall

In [ ]:
# Endereço da pasta de dados do projeto no GitHub (arquivos "raw" = texto puro).
# O pandas consegue ler um CSV direto de um endereço da internet.
URL_DADOS = 'https://raw.githubusercontent.com/fcoliveira-utfpr/change_climate/main/code/dados'

In [ ]:
import pandas as pd

# Lê as séries mensais (1961-2025) direto do GitHub.
#   parse_dates=['data'] -> a coluna 'data' é entendida como data
#   index_col='data'     -> a coluna 'data' vira o índice (o "nome" de cada linha)
serie_sh = pd.read_csv(f'{URL_DADOS}/santa_helena_mensal.csv', parse_dates=['data'], index_col='data')
serie_ct = pd.read_csv(f'{URL_DADOS}/curitiba_mensal.csv', parse_dates=['data'], index_col='data')

LATITUDE_SH = -24.86
LATITUDE_CT = -25.43

serie_sh.head()

## 1. As funções
Três funções: `folga_mais_proxima` (distância ao limite de classe mais próximo), `clima_de_um_periodo` (uma
janela) e `analisar_ponto` (todas as janelas de 1990 a 2025). Leia os comentários de cada uma.

In [ ]:
import agrometeorologiapy as amp
import pandas as pd

# Limites de classe de Thornthwaite (os mesmos da biblioteca)
LIMITES_IM = [-40, -20, 0, 20, 40, 60, 80, 100]                 # umidade, em %
LIMITES_ETP = [142, 285, 427, 570, 712, 855, 997, 1140]        # térmica, em mm


def folga_mais_proxima(valor, limites):
    """Distância entre "valor" e o limite de classe mais próximo dele."""
    menor_distancia = 99999
    for limite in limites:
        distancia = abs(valor - limite)
        if distancia < menor_distancia:
            menor_distancia = distancia
    return menor_distancia


def clima_de_um_periodo(serie, primeiro_ano, ultimo_ano, latitude):
    """Normal de um período (média de cada mês) -> classes de Köppen e Thornthwaite e folgas."""
    # 1) Recorta o período e calcula a normal
    janela = serie.loc[str(primeiro_ano):str(ultimo_ano)]
    normal = janela.groupby(janela.index.month).mean()
    temperatura = normal['tmed'].tolist()
    chuva = normal['chuva'].tolist()
    chuva_anual = sum(chuva)

    # 2) Köppen
    koppen = amp.classificacao_koppen(temperatura, chuva, latitude)

    # 3) Thornthwaite
    tabela_etp = amp.thornthwaite_mensal(pd.DataFrame({'T_media_C': temperatura}), lat=latitude)
    etp = tabela_etp['ETP_mm_mes']
    thornthwaite = amp.classificacao_thornthwaite(chuva, etp, latitude)

    # 4) Junta tudo num dicionário
    resultado = {
        'periodo': str(primeiro_ano) + '-' + str(ultimo_ano),
        'koppen': koppen['classe'],
        'thornthwaite': thornthwaite['classe'],
        'folga_A': min(temperatura) - 18,
        'folga_a': max(temperatura) - 22,
        'folga_f': min(chuva) - 60,
        'folga_m': min(chuva) - (100 - chuva_anual / 25),
        'Im': thornthwaite['Im'],
        'folga_Im': folga_mais_proxima(thornthwaite['Im'], LIMITES_IM),
        'ETP_anual': thornthwaite['ETP_anual'],
        'folga_ETP': folga_mais_proxima(thornthwaite['ETP_anual'], LIMITES_ETP),
    }
    return resultado


def analisar_ponto(serie, latitude, primeiro=1990, ultimo=2025):
    """Clima de referência móvel: uma linha por ano, com a normal dos 30 anos que terminam nele."""
    linhas = []
    for ano in range(primeiro, ultimo + 1):
        linha = clima_de_um_periodo(serie, ano - 29, ano, latitude)
        linha['ano'] = ano
        linhas.append(linha)
    tabela = pd.DataFrame(linhas)
    tabela = tabela.set_index('ano')
    return tabela

In [ ]:
# Teste com valores de resposta conhecida
print(folga_mais_proxima(72.4, LIMITES_IM))     # deve dar 7.6 (o limite mais próximo é 80)
print(folga_mais_proxima(1000, LIMITES_ETP))    # deve dar 3 (o mais próximo é 997)

## 2. Agora são duas linhas por cidade

In [ ]:
resultado_sh = analisar_ponto(serie_sh, LATITUDE_SH)
resultado_ct = analisar_ponto(serie_ct, LATITUDE_CT)
resultado_ct[['koppen', 'thornthwaite', 'folga_a', 'Im', 'folga_Im']].round(2)

## 3. Um laço sobre uma **lista de lugares** e uma tabela-resumo
Aqui a lista tem só as duas cidades. Na seção **Sua vez**, ela vira a lista das 27 capitais.

In [ ]:
# Cada lugar é um dicionário com nome, arquivo e latitude
lugares = [
    {'nome': 'Santa Helena', 'arquivo': 'santa_helena_mensal.csv', 'lat': -24.86},
    {'nome': 'Curitiba', 'arquivo': 'curitiba_mensal.csv', 'lat': -25.43},
]

linhas_resumo = []
for lugar in lugares:
    # 1) lê a série do lugar
    serie = pd.read_csv(f"{URL_DADOS}/{lugar['arquivo']}", parse_dates=['data'], index_col='data')

    # 2) janela móvel completa
    resultado = analisar_ponto(serie, lugar['lat'])

    # 3) quantas vezes a classe de Köppen mudou de um ano para o outro
    mudou = resultado['koppen'] != resultado['koppen'].shift()
    numero_de_trocas = mudou.iloc[1:].sum()

    # 4) uma linha de resumo: o clima de 1990 (1961-1990) e o de 2025 (1996-2025)
    linhas_resumo.append({
        'lugar': lugar['nome'],
        'koppen_1990': resultado.loc[1990, 'koppen'],
        'koppen_2025': resultado.loc[2025, 'koppen'],
        'trocas_koppen': numero_de_trocas,
        'thornthwaite_1990': resultado.loc[1990, 'thornthwaite'],
        'thornthwaite_2025': resultado.loc[2025, 'thornthwaite'],
        'folga_A_1990': resultado.loc[1990, 'folga_A'],
        'folga_A_2025': resultado.loc[2025, 'folga_A'],
        'folga_a_1990': resultado.loc[1990, 'folga_a'],
        'folga_a_2025': resultado.loc[2025, 'folga_a'],
    })

resumo = pd.DataFrame(linhas_resumo)
resumo.round(2)

## ✍️ Sua vez: as 27 capitais
1. Troque a lista `lugares` pela tabela das capitais. Dica: use `for indice, capital in capitais.iterrows():`,
   o arquivo `f"{BASE}/dados/capitais/{capital['uf'].lower()}.csv"` (as séries que você extraiu na
   Missão 5) e a latitude `capital['lat']`.
2. Acrescente ao resumo as colunas `regiao` e `lon` (vão servir para o mapa).
3. Salve o resumo em `resultados/resumo_27_capitais.csv` no seu Drive.
4. **Mapa simples:** faça um gráfico de dispersão (`plt.scatter`) com `lon` no eixo x, `lat` no eixo y e a
   cor dada pela `folga_A_2025` (use `c=...` e `cmap='coolwarm'`).
5. Quantas capitais mudaram de classe de Köppen entre 1990 e 2025? Quais oscilaram (mais de 1 troca)?

In [ ]:
# As capitais que VOCÊ extraiu na Missão 5 (exemplo 06) estão no seu Drive.
from google.colab import drive
drive.mount('/content/drive')
BASE = '/content/drive/MyDrive/mudanca_climatica_br'

# Tabela das 27 capitais (uf, capital, regiao, lat, lon), montada por você
capitais = pd.read_csv(f'{BASE}/dados/capitais_27.csv')

# Exemplo: lendo a série de uma capital (sigla em minúsculas)
uf = 'df'
serie_capital = pd.read_csv(f'{BASE}/dados/capitais/{uf}.csv', parse_dates=['data'], index_col='data')
serie_capital.head()